In [1]:
import pandas as pd
import json
import re
from pathlib import Path

# V2.3 Ingredient Standardization

## Objective
Prepare the ingredient data so it can be used for similarity modeling and TF-IDF analysis.

The goal of this stage is to transform the raw ingredient strings into standardized ingredient tokens.

This notebook will:
1. Load the processed skincare dataset
2. Inspect the ingredient column
3. Tokenize ingredient strings
4. Normalize ingredient names
5. Map synonyms to canonical ingredient names
6. Create a cleaned ingredient representation per product
7. Export the cleaned ingredient dataset

In [2]:
df = pd.read_csv("data/products_dataset_processed.csv")

print("Rows:", len(df))
df.head()

Rows: 50305


,product_name,brand,usage_type,category,ingredients,image_url,product_url,price
0,no7 hydra luminous aqua release skin perfector...,no7,Face,Tinted Moisturizers,"propylene glycol, alcohol denat, anhydroxylito...",https://cdn1.skinsafeproducts.com/photo/E84797...,https://www.skinsafeproducts.com/no7-hydra-lum...,15.29
1,"tower 28 sos daily rescue facial spray, 4 fl o...",tower 28,Face,Spray Moisturizer,"hypochlorous acid, sodium chloride, water",https://cdn1.skinsafeproducts.com/photo/488026...,https://www.skinsafeproducts.com/tower-28-sos-...,5.48
2,"clinique takes the day off cleansing oil, 6.7 ...",clinique,Facial Cleansers,Facial Cleansing Oil,"bht, caprylyl glycol, cetyl ethylhexanoate, gl...",https://cdn1.skinsafeproducts.com/photo/53EEB4...,https://www.skinsafeproducts.com/clinique-take...,43.95
3,"necessaire the body lotion, fragrance free, 15...",necessaire,Body,Lotions,"aqua/water/eau, caprylhydroxamic acid, capryli...",https://cdn1.skinsafeproducts.com/photo/D9ADD5...,https://www.skinsafeproducts.com/necessaire-th...,13.48
4,paula’s choice resist weightless advanced repa...,paula's choice,Toners & Astringents,Toners,"acetyl glucosamine, adenosine, butylene glycol...",https://cdn1.skinsafeproducts.com/photo/2A9B12...,https://www.skinsafeproducts.com/paula-s-choic...,18.00


## 1. Inspect the ingredient column

Before designing the cleaning pipeline, we inspect the ingredient column to understand:
- how ingredients are formatted
- whether ingredients are comma-separated
- whether there are problematic patterns such as slashes, parentheses, or internal commas

In [17]:
df["ingredients"].dropna().sample(10, random_state=42).tolist()

['citrus limonum (lemon) extract, helianthus annuus (sunflower) seed oil, sesamum indicum (sesame) seed oil, tocopheryl acetate (coconut derived), triticum vulgare (wheat) germ oil, vitis vinifera (grape) seed oil',
 'allafia, allantoin, aloe vera gel, black walnut, canola oil, castor oil, chamomille, cherry bark, comfrey burdock root, cyclomethicone, diazolidinyl urea, dicettldimonium chloride, dl-panthenol (vitamin b), fragrance, ginseng, glycerin, henna, hyssop, iodopropynyl butylcarbamate, mineral oil, olive oil, polyquaternium-32, propylene glycol, purified water, rumex acetosella (red sorrel) extract, sage, sesame oil, slippery elm bark, sunflower oil, sweet almond oil, vitamin a, vitamin d, vitamin e, wheat germ, wheat germ oil, yarrow',
 '1, 2-hexanediol, allantoin, anhydroxylitol, artemisia capillaris extract, artemisia vulgaris oil, benzyl glycol, betula platyphylla japonica juice, butylene glycol, caprylyl glycol, centella asiatica extract*, chamaecyparis obtusa water, citri

## 2. First tokenization pass

As a first baseline, ingredient strings are split into tokens.
However, because some ingredients contain internal commas, this first pass must be checked carefully.

In [16]:
def smart_split_ingredients(ingredients):
    if not isinstance(ingredients, str) or not ingredients.strip():
        return []

    text = ingredients.strip().lower()

    # protect numeric ingredient names like "1, 2-hexanediol"
    text = re.sub(r'(\d)\s*,\s*(\d-\w)', r'\1@@\2', text)

    parts = [p.strip() for p in text.split(",") if p.strip()]

    tokens = []
    seen = set()

    for p in parts:
        t = p.replace("@@", ", ")
        t = re.sub(r"\s+", " ", t).strip()

        if t and t not in seen:
            tokens.append(t)
            seen.add(t)

    return tokens

In [18]:
sample = df["ingredients"].dropna().sample(5, random_state=1)

for ing in sample:
    print("ORIGINAL:", ing)
    print("TOKENS:", smart_split_ingredients(ing))
    print("-" * 80)

ORIGINAL: cetostearyl alcohol, chlorocresol, liquid paraffin, macrogol cetostearyl ether, phosphoric acid, purified water, sodium dihydrogen phosphate, white soft paraffin
TOKENS: ['cetostearyl alcohol', 'chlorocresol', 'liquid paraffin', 'macrogol cetostearyl ether', 'phosphoric acid', 'purified water', 'sodium dihydrogen phosphate', 'white soft paraffin']
--------------------------------------------------------------------------------
ORIGINAL: benzophenone-4, citric acid, cocamide mipa, cocamidopropyl betaine, d&c violet no. 2, fragrance, laureth-4, methylchloroisothiazolinone, methylisothiazolinone, polyquaternium-7, sodium chloride, sodium laureth sulfate, tetrasodium edta, tocopheryl acetate, water
TOKENS: ['benzophenone-4', 'citric acid', 'cocamide mipa', 'cocamidopropyl betaine', 'd&c violet no. 2', 'fragrance', 'laureth-4', 'methylchloroisothiazolinone', 'methylisothiazolinone', 'polyquaternium-7', 'sodium chloride', 'sodium laureth sulfate', 'tetrasodium edta', 'tocopheryl ac

In [19]:
df["ingredient_tokens"] = df["ingredients"].apply(smart_split_ingredients)

df[["ingredients", "ingredient_tokens"]].head()

,ingredients,ingredient_tokens
0,"propylene glycol, alcohol denat, anhydroxylito...","[propylene glycol, alcohol denat, anhydroxylit..."
1,"hypochlorous acid, sodium chloride, water","[hypochlorous acid, sodium chloride, water]"
2,"bht, caprylyl glycol, cetyl ethylhexanoate, gl...","[bht, caprylyl glycol, cetyl ethylhexanoate, g..."
3,"aqua/water/eau, caprylhydroxamic acid, capryli...","[aqua/water/eau, caprylhydroxamic acid, capryl..."
4,"acetyl glucosamine, adenosine, butylene glycol...","[acetyl glucosamine, adenosine, butylene glyco..."


## 3. Check whether synonym normalization is needed

The previous dataset we used intially used a synonym dictionary we built.
Before reusing it here, we first check whether the current dataset actually contains ingredient variants or misspellings covered by that dictionary.

In [20]:
with open("synonyms.json", "r", encoding="utf-8") as f:
    synonyms = json.load(f)

list(synonyms.items())[:10]

[('alchol denat', 'alcohol denat'),
 ('denat', 'alcohol denat'),
 ('sd alcohol 40 b', 'alcohol denat'),
 ('sd alcohol 40-b', 'alcohol denat'),
 ('acsorbyl palmitate', 'ascorbyl palmitate'),
 ('ascorbyl glu-coside', 'ascorbyl glucoside'),
 ('ascorbyl tetraisopal-mitate', 'ascorbyl tetraisopalmitate'),
 ('caffine', 'caffeine'),
 ('camelia sinensis', 'camellia sinensis'),
 ('ca-mellia sinensis', 'camellia sinensis')]

In [21]:
synonym_keys = set(synonyms.keys())
tokens = df["ingredient_tokens"].explode()

tokens[tokens.isin(synonym_keys)].value_counts().head(20)

ingredient_tokens
1                                                     253
sd alcohol 40-b                                       104
citricacid                                             20
amica montana flower extract                           11
citrus auranitum bergamia (bergamot) fruit extract      5
butylene gylcol                                         3
sd alcohol 40 b                                         2
Name: count, dtype: int64

## Observation

The synonym diagnostic shows that the current dataset still contains a small number of ingredient spelling variants and malformed tokens.

The most frequent issues include:
- `sd alcohol 40-b`
- `citricacid`
- `amica montana flower extract`
- `butylene gylcol`

This confirms that synonym normalization is still useful, although on this dataset it plays a smaller role than in the previous project.

The largest remaining artifact is the token `1`, which suggests that some ingredients such as `1, 2-hexanediol` are still being split imperfectly.

In [ ]:
with open("ingredient_synonyms_v2.json", "r", encoding="utf-8") as f:
    synonyms = json.load(f)

## 4. Apply synonym normalization

After identifying spelling variants and malformed tokens that still appear in the dataset, synonym normalization is applied to map these variants to their canonical forms.

At this stage, the goal is not yet to collapse broader ingredient families such as `water/aqua/eau`, but simply to correct direct spelling variants and malformed tokens.

In [26]:
with open("synonyms_v2.json", "r", encoding="utf-8") as f:
    synonyms = json.load(f)

list(synonyms.items())[:10]

[('alchol denat', 'alcohol denat'),
 ('sd alcohol 40 b', 'alcohol denat'),
 ('sd alcohol 40-b', 'alcohol denat'),
 ('acsorbyl palmitate', 'ascorbyl palmitate'),
 ('ascorbyl glu-coside', 'ascorbyl glucoside'),
 ('ascorbyl tetraisopal-mitate', 'ascorbyl tetraisopalmitate'),
 ('caffine', 'caffeine'),
 ('camelia sinensis', 'camellia sinensis'),
 ('ca-mellia sinensis', 'camellia sinensis'),
 ('camillia sinensis', 'camellia sinensis')]

In [27]:
def apply_synonyms(tokens, synonym_map):
    mapped = []
    seen = set()

    for t in tokens:
        new = synonym_map.get(t, t)

        if new and new not in seen:
            mapped.append(new)
            seen.add(new)

    return mapped

In [28]:
df["ingredient_tokens_syn"] = df["ingredient_tokens"].apply(
    lambda x: apply_synonyms(x, synonyms)
)

In [29]:
tokens_syn = df["ingredient_tokens_syn"].explode()
synonym_keys = set(synonyms.keys())

tokens_syn[tokens_syn.isin(synonym_keys)].value_counts().head(20)

Series([], Name: count, dtype: int64)

## 5. Canonical ingredient mapping

After typo and synonym normalization, a second normalization stage is applied to collapse equivalent ingredient variants into a smaller set of canonical ingredient names.

This is especially useful for:
- water variants (`aqua`, `eau`, `water/aqua/eau`)
- fragrance variants (`parfum`, `fragrance/parfum`, `perfume`)
- ingredient families such as vitamin E forms and common pigments

This improves consistency before TF-IDF is applied.

In [30]:
tokens_syn[tokens_syn.str.contains(r"water|aqua|eau", case=False, na=False)].value_counts().head(30)

ingredient_tokens_syn
water\aqua\eau                                  8568
water                                           7878
water (aqua)                                    4331
aqua                                            3235
water/aqua/eau                                  3145
aqua (water)                                    2841
aqua/water/eau                                  2812
eau)                                            2479
water (aqua                                     1941
hamamelis virginiana (witch hazel) water         812
aqua/water                                       687
purified water                                   675
aqua (water                                      546
water (aqua/eau)                                 499
water/eau                                        489
rosa damascena flower water                      488
citrullus lanatus (watermelon) fruit extract     435
aqua / water / eau                               368
water/aqua              

In [31]:
tokens_syn[tokens_syn.str.contains(r"parfum|fragrance|perfume|aroma|flavor", case=False, na=False)].value_counts().head(30)

ingredient_tokens_syn
fragrance                           7317
fragrance (parfum)                  4736
parfum (fragrance)                  2402
parfum                              1885
perfume                             1585
parfum/fragrance                    1478
fragrance/parfum                     972
flavor                               553
parfum / fragrance                   410
fragrance / parfum                   253
natural fragrance                    228
aroma                                187
fragrance(parfum)                    176
flavor (aroma)                       170
flavor/aroma                          84
aroma (flavor)                        79
fragrance (essential oil blend)       74
fragrance allergens not removed)      66
aroma/flavor                          66
parfum/ fragrance                     63
fragrance (natural)                   52
flavors                               51
fragrance/ parfum                     44
fragrance (parfum )                

In [32]:
tokens_syn[tokens_syn.str.contains(r"tocopherol|tocopheryl|tocotrienol", case=False, na=False)].value_counts().head(30)

ingredient_tokens_syn
tocopheryl acetate                                                 11288
tocopherol                                                          9778
tocopherol (multisource)                                            2540
tocopherol (vitamin e)                                              1058
tocopheryl acetate (vitamin e)                                       798
tocopheryl acetate (coconut derived)                                 344
tocotrienols                                                         160
tocopherol acetate                                                   127
tocopheryl acetate (vitamin e acetate)                               110
mixed tocopherols (vitamin e)                                         99
tocopheryl (vitamin e) acetate                                        87
tocopheryl succinate                                                  83
tocopherol acetate (vitamin e)                                        82
d-alpha tocopherol acetate (m

In [33]:
tokens_syn[tokens_syn.str.contains(r"mica|ci 77019|titanium dioxide|ci 77891|iron oxide|ci 77491|ci 77492|ci 77499", case=False, na=False)].value_counts().head(40)

ingredient_tokens_syn
titanium dioxide               2759
mica                           2328
titanium dioxide (ci 77891)    2208
ci 77492                       1628
iron oxides (ci 77491          1403
ci 77499)                      1349
ci 77891 (titanium dioxide)     642
iron oxides                     466
iron oxides (ci 77491)          302
ci 77891                        270
ci 77491                        267
ci 77499                        183
ci 77891/titanium dioxide       160
ci 77891 / titanium dioxide     157
iron oxides ci 77491            142
mica (ci 77019)                 139
ci 77491 (iron oxides)          109
iron oxides (ci 77492)          107
ci 77499 (iron oxides)           95
iron oxides (ci 77499)           76
iron oxide                       71
ci 77492)                        64
iron oxides (ci 77492            62
ci 77492 (iron oxides)           57
iron oxide (ci 77491)            55
iron oxide red (ci 77491)        39
ci 77499).                       39
ci 774

In [34]:
tokens_syn[tokens_syn.str.contains(r"citrus|orange|lemon|lime|grapefruit|bergamot|mandarin|tangerine|yuzu|aurantium|sinensis|limon|junos|reticulata|bergamia", case=False, na=False)].value_counts().head(50)

ingredient_tokens_syn
limonene                                                 8763
camellia sinensis leaf extract                           2148
camellia sinensis (green tea) leaf extract               1488
citrus aurantium dulcis (orange) peel oil                 838
citrus aurantium bergamia (bergamot) fruit oil            628
citrus aurantium dulcis (orange) fruit extract            598
citrus limonum (lemon) extract                            572
citrus limon (lemon) fruit extract                        509
citrus limon (lemon) peel oil                             468
citrus grandis (grapefruit) peel oil                      389
citrus nobilis (mandarin orange) peel oil                 308
citrus aurantium dulcis (orange) oil                      304
citrus paradisi (grapefruit) peel oil                     282
citrus aurantium dulcis (orange peel) oil                 249
citrus grandis (grapefruit) fruit extract                 235
citrus aurantifolia (lime) oil                  

In [43]:
CANON_RULES_SMALL = [
    ("water", [
        r"^\s*aqua\s*$",
        r"^\s*water\s*$",
        r"^\s*eau\s*$",
        r"aqua\s*/\s*water\s*/\s*eau",
        r"water\s*/\s*aqua\s*/\s*eau",
        r"water\s*\\\s*aqua\s*\\\s*eau",
        r"aqua\s*\\\s*water\s*\\\s*eau",
        r"water/aqua/eau",
        r"aqua/water/eau",
        r"water\\aqua\\eau",
        r"water\s*\(aqua\)",
        r"aqua\s*\(water\)",
        r"water\s*\(aqua/eau\)",
        r"aqua\s*\(water/eau\)",
        r"water/eau",
        r"water/aqua",
        r"aqua/water",
        r"purified\s+water",
        r"^water\s*\(aqua$",
        r"^eau\)$"
    ]),

    ("fragrance", [
        r"^\s*parfum\s*$",
        r"^\s*fragrance\s*$",
        r"^\s*perfume\s*$",
        r"parfum\s*\(fragrance\)",
        r"fragrance\s*\(parfum\)",
        r"fragrance\s*/\s*parfum",
        r"parfum\s*/\s*fragrance",
        r"parfum/fragrance",
        r"fragrance/parfum",
        r"natural\s+fragrance",
        r"^\s*aroma\s*$",
        r"^\s*flavor\s*$",
        r"^\s*flavors\s*$",
        r"flavor\s*\(aroma\)",
        r"aroma\s*\(flavor\)",
        r"flavor/aroma",
        r"aroma/flavor"
    ]),

    ("vitamin e", [
        r"\btocopherol\b",
        r"\btocopheryl\s+acetate\b",
        r"\btocopheryl\s+succinate\b",
        r"\btocotrienols\b",
        r"\btocopherol\s*\(vitamin e\)",
        r"\btocopheryl acetate\s*\(vitamin e\)",
        r"\btocopherol\s*\(natural vitamin e\)",
        r"\bmixed tocopherols\b",
        r"\btocopherols\b",
        r"\btocopheryl\b"
    ]),

    ("mica", [
        r"\bmica\b",
        r"\bmica\s*\(ci\s*77019\)",
        r"\bci\s*77019\b",
        r"\bci\s*77019\s*\(mica\)"
    ]),

    ("titanium dioxide", [
        r"titanium\s+dioxide",
        r"titanium dioxide\s*\(ci\s*77891\)",
        r"ci\s*77891\s*\(titanium dioxide\)",
        r"ci\s*77891/titanium dioxide",
        r"ci\s*77891\s*/\s*titanium dioxide",
        r"\bci\s*77891\b"
    ]),

    ("iron oxides", [
        r"iron\s+oxides",
        r"iron oxide",
        r"iron oxides\s*\(ci\s*77491\)?",
        r"iron oxides\s*\(ci\s*77492\)?",
        r"iron oxides\s*\(ci\s*77499\)?",
        r"ci\s*77491\s*\(iron oxides\)",
        r"ci\s*77492\s*\(iron oxides\)",
        r"ci\s*77499\s*\(iron oxides\)",
        r"iron oxides ci\s*77491",
        r"ci\s*77491/iron oxides",
        r"\bci\s*77491\b",
        r"\bci\s*77492\b",
        r"\bci\s*77499\b"
    ]),
]

In [44]:
CANON_RULES_SMALL_COMPILED = [
    (canon, [re.compile(pat, flags=re.IGNORECASE) for pat in pats])
    for canon, pats in CANON_RULES_SMALL
]

In [45]:
def apply_canon_to_tokens(tokens, canon_rules_compiled):
    if not isinstance(tokens, list):
        return []

    out = []
    seen = set()

    for tok in tokens:
        t = tok.strip()
        if not t:
            continue

        canon = None
        for canon_token, patterns in canon_rules_compiled:
            if any(p.search(t) for p in patterns):
                canon = canon_token
                break

        final_tok = canon if canon else t

        if final_tok not in seen:
            out.append(final_tok)
            seen.add(final_tok)

    return out

In [46]:
df["ingredient_tokens_clean"] = df["ingredient_tokens_syn"].apply(
    lambda toks: apply_canon_to_tokens(toks, CANON_RULES_SMALL_COMPILED)
)

## 6. Final cleaned ingredient dataset

After tokenization, synonym normalization, and canonical mapping, the cleaned ingredient tokens are stored in the final dataset.

This dataset will be used as the ingredient-based input for later TF-IDF analysis and recommendation modeling.

In [47]:
df_final = df.copy()

df_final[["brand", "product_name", "ingredients", "ingredient_tokens_clean"]].head(10)

,brand,product_name,ingredients,ingredient_tokens_clean
0,no7,no7 hydra luminous aqua release skin perfector...,"propylene glycol, alcohol denat, anhydroxylito...","[propylene glycol, alcohol denat, anhydroxylit..."
1,tower 28,"tower 28 sos daily rescue facial spray, 4 fl o...","hypochlorous acid, sodium chloride, water","[hypochlorous acid, sodium chloride, water]"
2,clinique,"clinique takes the day off cleansing oil, 6.7 ...","bht, caprylyl glycol, cetyl ethylhexanoate, gl...","[bht, caprylyl glycol, cetyl ethylhexanoate, g..."
3,necessaire,"necessaire the body lotion, fragrance free, 15...","aqua/water/eau, caprylhydroxamic acid, capryli...","[water, caprylhydroxamic acid, caprylic/capric..."
4,paula's choice,paula’s choice resist weightless advanced repa...,"acetyl glucosamine, adenosine, butylene glycol...","[acetyl glucosamine, adenosine, butylene glyco..."
5,aestura,"aestura atobarrier365 moisturizing cream, cera...","acrylates/ammonium methacrylate copolymer, all...","[acrylates/ammonium methacrylate copolymer, al..."
6,clinique,"clinique all about eyes cream, vitamin c, 1 oz...","butylene glycol, caffeine, camellia sinensis (...","[butylene glycol, caffeine, camellia sinensis ..."
7,the ordinary,"the ordinary a high-strength serum, alpha arbu...","alpha-arbutin, aqua (water), chlorphenesin, ci...","[alpha-arbutin, water, chlorphenesin, citric a..."
8,mario badescu skin care,"mario badescu skin care deep blemish solution,...","propylene glycol, allantoin, biotin, citric ac...","[propylene glycol, allantoin, biotin, citric a..."
9,roc,"roc line smoothing eye cream, retinol correxio...","ascorbic acid, bht, c12-15 alkyl benzoate, car...","[ascorbic acid, bht, c12-15 alkyl benzoate, ca..."


In [48]:
from pathlib import Path

PROCESSED_PATH = Path("data") / "products_dataset_clean_tokens.csv"
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

df_final.to_csv(PROCESSED_PATH, index=False)
print(f"Processed dataset saved to: {PROCESSED_PATH.resolve()}")

Processed dataset saved to: /Users/sabinabacaoanu/SkinCares/miguellib/datasets/data/products_dataset_clean_tokens.csv
